In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("BookPosCash") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/17 18:11:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
pos_cash = spark.read \
    .parquet("/data/processed/pos_cash")

pos_cash.createOrReplaceTempView("pos_cash")

# Contagem de linhas e colunas
num_rows = pos_cash.count()
num_columns = len(pos_cash.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

pos_cash.show(5, truncate=False)

Quantidade de linhas: 10001358
Quantidade de variaveis (colunas): 9
+----------+----------+----------------------+-----------------------------+---------------------------+--------------+------------------+-------------------+--------------+
|PK_AGG_POS|SK_ID_CURR|FVL_CNT_INSTALMENT_POS|FVL_CNT_INSTALMENT_FUTURE_POS|FC_NAME_CONTRACT_STATUS_POS|FVL_SK_DPD_POS|FVL_SK_DPD_DEF_POS|PK_DATREF_POS      |PK_DATPART_POS|
+----------+----------+----------------------+-----------------------------+---------------------------+--------------+------------------+-------------------+--------------+
|2137736   |355067    |6.0                   |0.0                          |Completed                  |0             |0                 |2022-12-01 00:00:00|202212        |
|2094457   |411079    |12.0                  |9.0                          |Active                     |0             |0                 |2022-12-01 00:00:00|202212        |
|1556649   |288150    |6.0                   |2.0             

In [3]:
qtd_linhas = pos_cash.count()

In [4]:
spark.sql("""
            Select
                FC_NAME_CONTRACT_STATUS_POS,
                count(*) as VOLUME,
                round(100*(count(*)/{}),2) as VOL_PERCENT
            from 
                pos_cash
            group by 
                FC_NAME_CONTRACT_STATUS_POS
            order by 
                VOLUME desc
""".format(qtd_linhas)).show(50,False)

+---------------------------+-------+-----------+
|FC_NAME_CONTRACT_STATUS_POS|VOLUME |VOL_PERCENT|
+---------------------------+-------+-----------+
|Active                     |9151119|91.5       |
|Completed                  |744883 |7.45       |
|Signed                     |87260  |0.87       |
|Demand                     |7065   |0.07       |
|Returned to the store      |5461   |0.05       |
|Approved                   |4917   |0.05       |
|Amortized debt             |636    |0.01       |
|Canceled                   |15     |0.0        |
|XNA                        |2      |0.0        |
+---------------------------+-------+-----------+



In [15]:
POS_01 = spark.sql("""
    SELECT
        PK_AGG_POS,

        -- Estatísticas gerais de valores dos créditos anteriores
        min(FVL_CNT_INSTALMENT_POS) as MIN_FVL_CNT_INSTALMENT_POS,
        max(FVL_CNT_INSTALMENT_POS) as MAX_FVL_CNT_INSTALMENT_POS,
        avg(FVL_CNT_INSTALMENT_POS) as AVG_FVL_CNT_INSTALMENT_POS,


        min(FVL_CNT_INSTALMENT_FUTURE_POS) as MIN_FVL_CNT_INSTALMENT_FUTURE_POS,
        max(FVL_CNT_INSTALMENT_FUTURE_POS) as MAX_FVL_CNT_INSTALMENT_FUTURE_POS,
        avg(FVL_CNT_INSTALMENT_FUTURE_POS) as AVG_FVL_CNT_INSTALMENT_FUTURE_POS,

        min(FVL_SK_DPD_POS) as MIN_FVL_SK_DPD_POS_POS,
        max(FVL_SK_DPD_POS) as MAX_FVL_SK_DPD_POS_POS,
        avg(FVL_SK_DPD_POS) as AVG_FVL_SK_DPD_POS_POS,
        
        min(FVL_SK_DPD_DEF_POS) as MIN_FVL_SK_DPD_DEF_POS_POS,
        max(FVL_SK_DPD_DEF_POS) as MAX_FVL_SK_DPD_DEF_POS_POS,
        avg(FVL_SK_DPD_DEF_POS) as AVG_FVL_SK_DPD_DEF_POS_POS,        

        --Quantidade de créditos por tipo
        sum(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' then 1 else 0 end) as QT_CONTRACT_STATUS_ACTIVE_POS,
        sum(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' then 1 else 0 end) as QT_CONTRACT_STATUS_COMPLETED_POS,


        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_ACTIVE_U3M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_ACTIVE_U6M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_ACTIVE_U9M_POS, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_ACTIVE_U12M_POS,


        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_COMPLETED_U3M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_COMPLETED_U6M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_COMPLETED_U9M_POS, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_POS else null end) as AVG_CNT_INSTALMENT_COMPLETED_U12M_POS,              

                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U3M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U6M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Active' and PK_DATPART_POS in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U9M_POS, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FVL_CNT_INSTALMENT_FUTURE_POS = 'Active' and PK_DATPART_POS in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U12M_POS,


        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U3M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U6M_POS,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_POS = 'Completed' and PK_DATPART_POS in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U9M_POS, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FVL_CNT_INSTALMENT_FUTURE_POS = 'Completed' and PK_DATPART_POS in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_FUTURE_POS else null end) as AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U12M_POS
                 

    

    FROM pos_cash
    GROUP BY PK_AGG_POS
""")

POS_01.createOrReplaceTempView("POS_01")
num_rows = POS_01.count()
num_columns = len(POS_01.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

[Stage 40:===================================>                    (12 + 6) / 19]

Quantidade de linhas: 936325
Quantidade de variaveis (colunas): 31


In [16]:
POS_01.show(5, truncate=False)

+----------+--------------------------+--------------------------+--------------------------+---------------------------------+---------------------------------+---------------------------------+----------------------+----------------------+----------------------+--------------------------+--------------------------+--------------------------+-----------------------------+--------------------------------+---------------------------------+---------------------------------+---------------------------------+----------------------------------+------------------------------------+------------------------------------+------------------------------------+-------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------------+-------------------------------------------+-------------------------------------------+-------------------------------------------+---------

In [17]:
POS_02 = spark.sql("""
    SELECT
        *,
        -- Razões entre médias por janelas de tempo (tendência temporal)
        round(AVG_CNT_INSTALMENT_ACTIVE_U3M_POS/AVG_CNT_INSTALMENT_ACTIVE_U6M_POS,2) as VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_ACTIVE_POS,
        round(AVG_CNT_INSTALMENT_ACTIVE_U6M_POS/AVG_CNT_INSTALMENT_ACTIVE_U9M_POS,2) as VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_ACTIVE_POS,
        round(AVG_CNT_INSTALMENT_ACTIVE_U9M_POS/AVG_CNT_INSTALMENT_ACTIVE_U12M_POS,2) as VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_ACTIVE_POS,
        
        round(AVG_CNT_INSTALMENT_COMPLETED_U3M_POS/AVG_CNT_INSTALMENT_COMPLETED_U6M_POS,2) as VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_COMPLETED_POS,
        round(AVG_CNT_INSTALMENT_COMPLETED_U6M_POS/AVG_CNT_INSTALMENT_COMPLETED_U9M_POS,2) as VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_COMPLETED_POS,
        round(AVG_CNT_INSTALMENT_COMPLETED_U9M_POS/AVG_CNT_INSTALMENT_COMPLETED_U12M_POS,2) as VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_COMPLETED_POS,


        round(AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U3M_POS/AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U6M_POS,2) as VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_FUTURE_ACTIVE_POS,
        round(AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U6M_POS/AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U9M_POS,2) as VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_FUTURE_ACTIVE_POS,
        round(AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U9M_POS/AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U12M_POS,2) as VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_FUTURE_ACTIVE_POS,

        round(AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U3M_POS/AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U6M_POS,2) as VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_FUTURE_COMPLETED_POS,
        round(AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U6M_POS/AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U9M_POS,2) as VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_FUTURE_COMPLETED_POS,
        round(AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U9M_POS/AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U12M_POS,2) as VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_FUTURE_COMPLETED_POS
        


    FROM POS_01

""")

POS_02.createOrReplaceTempView("POS_02")
num_rows = POS_02.count()
num_columns = len(POS_02.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

Quantidade de linhas: 936325
Quantidade de variaveis (colunas): 43


In [18]:
POS_02.show(5, truncate=False)

+----------+--------------------------+--------------------------+--------------------------+---------------------------------+---------------------------------+---------------------------------+----------------------+----------------------+----------------------+--------------------------+--------------------------+--------------------------+-----------------------------+--------------------------------+---------------------------------+---------------------------------+---------------------------------+----------------------------------+------------------------------------+------------------------------------+------------------------------------+-------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------------+-------------------------------------------+-------------------------------------------+-------------------------------------------+---------

# 📘 Dicionário de Variáveis - Book POS_CASH

| Variável | Descrição |
|----------|-----------|
| `MIN_FVL_CNT_INSTALMENT_POS` | Número mínimo de parcelas de créditos anteriores |
| `MAX_FVL_CNT_INSTALMENT_POS` | Número máximo de parcelas de créditos anteriores |
| `AVG_FVL_CNT_INSTALMENT_POS` | Número médio de parcelas de créditos anteriores |
| `MIN_FVL_CNT_INSTALMENT_FUTURE_POS` | Número mínimo de parcelas restantes a pagar em créditos anteriores |
| `MAX_FVL_CNT_INSTALMENT_FUTURE_POS` | Número máximo de parcelas restantes a pagar em créditos anteriores |
| `AVG_FVL_CNT_INSTALMENT_FUTURE_POS` | Número médio de parcelas restantes a pagar em créditos anteriores |
| `MIN_FVL_SK_DPD_POS_POS` | Número mínimo de dias em atraso (DPD) em créditos anteriores |
| `MAX_FVL_SK_DPD_POS_POS` | Número máximo de dias em atraso (DPD) em créditos anteriores |
| `AVG_FVL_SK_DPD_POS_POS` | Número médio de dias em atraso (DPD) em créditos anteriores |
| `MIN_FVL_SK_DPD_DEF_POS_POS` | Número mínimo de dias em atraso com tolerância (DPD_DEF) em créditos anteriores |
| `MAX_FVL_SK_DPD_DEF_POS_POS` | Número máximo de dias em atraso com tolerância (DPD_DEF) em créditos anteriores |
| `AVG_FVL_SK_DPD_DEF_POS_POS` | Número médio de dias em atraso com tolerância (DPD_DEF) em créditos anteriores |
| `QT_CONTRACT_STATUS_ACTIVE_POS` | Quantidade de créditos anteriores com status 'Active' |
| `QT_CONTRACT_STATUS_COMPLETED_POS` | Quantidade de créditos anteriores com status 'Completed' |
| `AVG_CNT_INSTALMENT_ACTIVE_U3M_POS` | Média de parcelas para créditos ativos nos últimos 3 meses |
| `AVG_CNT_INSTALMENT_ACTIVE_U6M_POS` | Média de parcelas para créditos ativos nos últimos 6 meses |
| `AVG_CNT_INSTALMENT_ACTIVE_U9M_POS` | Média de parcelas para créditos ativos nos últimos 9 meses |
| `AVG_CNT_INSTALMENT_ACTIVE_U12M_POS` | Média de parcelas para créditos ativos nos últimos 12 meses |
| `AVG_CNT_INSTALMENT_COMPLETED_U3M_POS` | Média de parcelas para créditos completados nos últimos 3 meses |
| `AVG_CNT_INSTALMENT_COMPLETED_U6M_POS` | Média de parcelas para créditos completados nos últimos 6 meses |
| `AVG_CNT_INSTALMENT_COMPLETED_U9M_POS` | Média de parcelas para créditos completados nos últimos 9 meses |
| `AVG_CNT_INSTALMENT_COMPLETED_U12M_POS` | Média de parcelas para créditos completados nos últimos 12 meses |
| `AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U3M_POS` | Média de parcelas restantes para créditos ativos nos últimos 3 meses |
| `AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U6M_POS` | Média de parcelas restantes para créditos ativos nos últimos 6 meses |
| `AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U9M_POS` | Média de parcelas restantes para créditos ativos nos últimos 9 meses |
| `AVG_CNT_INSTALMENT_FUTURE_ACTIVE_U12M_POS` | Média de parcelas restantes para créditos ativos nos últimos 12 meses |
| `AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U3M_POS` | Média de parcelas restantes para créditos completados nos últimos 3 meses |
| `AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U6M_POS` | Média de parcelas restantes para créditos completados nos últimos 6 meses |
| `AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U9M_POS` | Média de parcelas restantes para créditos completados nos últimos 9 meses |
| `AVG_CNT_INSTALMENT_FUTURE_COMPLETED_U12M_POS` | Média de parcelas restantes para créditos completados nos últimos 12 meses |
| `VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_ACTIVE_POS` | Razão entre médias de parcelas ativas (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_ACTIVE_POS` | Razão entre médias de parcelas ativas (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_ACTIVE_POS` | Razão entre médias de parcelas ativas (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_COMPLETED_POS` | Razão entre médias de parcelas completadas (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_COMPLETED_POS` | Razão entre médias de parcelas completadas (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_COMPLETED_POS` | Razão entre médias de parcelas completadas (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_FUTURE_ACTIVE_POS` | Razão entre médias de parcelas futuras ativas (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_FUTURE_ACTIVE_POS` | Razão entre médias de parcelas futuras ativas (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_FUTURE_ACTIVE_POS` | Razão entre médias de parcelas futuras ativas (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_FUTURE_COMPLETED_POS` | Razão entre médias de parcelas futuras completadas (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_FUTURE_COMPLETED_POS` | Razão entre médias de parcelas futuras completadas (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_FUTURE_COMPLETED_POS` | Razão entre médias de parcelas futuras completadas (9m/12m) |

#### Salvando tabela particionada (Parquet)

In [19]:
nm_path = '/data/books/pos_cash'
POS_02.write.parquet(nm_path, mode='overwrite')
#bureau_etl_01.coalesce(1).write.parquet(nm_path, mode="overwrite")

In [20]:
spark.stop()